# Datamine SLIPER Process Example

This notebook demonstrates how to configure and run the **`sliper`** process wrapper in `dmstudio`.

## Process Description

## SLIPER

Intersects a set of parallel perimeters by a set of orthogonal planes.

This process produces two optional output files:

1. An intersection file, containing the co-ordinates of the end points of the line describing the intersection of each plane and the perimeters.

2. A perimeter file, connecting the co-ordinates of the intersections in the form of a standard perimeter.

The input perimeters may be parallel to any one of the three co-ordinate axes, and the intersection plane will be perpendicular to the input perimeter.

For example, Figure 1 shows a set on input perimeters representing the plan outline of a particular rock type on 4 benches. These perimeters are intersected by 2 north-south planes. The output intersections file contains 7 fields - the X, Y and Z co-ordinates of the 2 end points (**X1, Y1, Z1** and **X2, Y2, Z2**) and a perimeter value field containing the **PVALUE** of the original perimeter. In the example of Figure 1 the file will contain 7 records, one for each of the intersection lines **P1P2** , **P3P4** , .... **P13P14**. The **X1** and **X2** values are northings, **Y1** and **Y2** the elevations, and **Z1** and **Z2** the easting of the intersection plane. The **PVALUE** for intersection line **P1P2** is 10.

The intersection lines for any intersection plane can be plotted by using the [PLOTLN](<plotln.md>) process under the retrieval criteria that **ZP** is equal to the section co-ordinate . In this example the retrieval criteria would be for the easting (ZP) value. Each intersection line can be annotated with the original **PVALUE** using the [PLOTAN](<plotan.md>) process. In this way it is possible for the input perimeter file to contain several perimeters for each bench, representing different rock types, with the PVALUE field containing a rocktype identifier. Using **PLOTLN** as described above it would be possible to distinguish between intersection lines for the different rock types.

The output perimeter file is a standard perimeter containing fields **XP, YP, ZP, PTN** and **PVALUE**. In the example of Figure 2, the output perimeter file would contain two perimeters. The first perimeter would have eight points ordered **P1, P3, P5, P7, P8, P6, P4, P2** and with both **ZP** and **PVALUE** equal to the section easting. If the input perimeter contains a re-entrant, or there are multiple perimeters on a single bench, then the output perimeter will describe the outermost points of the intersections of any line.

* **Note** (There is a limit of 1200 points in any single input or output perimeter, and a limit of 20 intersection lines, for the intersection of a single perimeter plane and an intersection plane.):

### Input Files:

* **perimin** (String):
  Input perimeter file containing at least 2 perimeters. This must contain the
  fields[**XP,YP,ZP,PTN,PVALUE** , all numeric and explicit] with **XP** and **YP**
  variable and **ZP** fixed for each perimeter. The file must contain at least two
  perimeters with different **ZP** values.
  Required=Yes

### Output Files:

* **intersec** (Undefined):
  An optional output intersections file containing the coordinates **X1,Y1,Z1** of the
  end points of each intersection line, and the **PVALUE** of the intersected perimeter.
  At least 1 output file must be specified.
  Required=No

* **perimout** (String):
  An optional output perimeter file containing the intersection perimeters. At least 1
  output file must be specified.
  Required=No

### Fields:

### Parameters:

* **mode**:

* **Options** (1: FOR A PLAN TO NS CONVERSION.; 2: FOR A PLAN TO EW CONVERSION.; 3: FOR A NS):
  TO EW CONVERSION.; 4: FOR A NS TO PLAN CONVERSION.; 5: FOR A EW TO NS CONVERSION.; 6:

## FOR A EW TO PLAN CONVERSION.

  Range=1,6
  Values=1,2,3,4,5,6
  Default=1
  Required=Yes

* **startpos**:
  Starting position for intersection planes.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **stize**:
  Interval between intersection planes.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **numst**:
  Number of intersection planes.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **close**:
  Close perimeters, either 1 to close or 0 to not close.
  Range=0,1
  Values=0,1
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('sliper')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute sliper
print("Running sliper...")
dm_cmd.sliper(
    perimin_i='t_input_file',  # required
    mode_p='required_val',  # required
    startpos_p='required_val',  # required
    stize_p='required_val',  # required
    numst_p='required_val',  # required
    # intersec_o='t_sliper_out',  # optional
    # perimout_o='t_sliper_out',  # optional
    # close_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("sliper execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_sliper_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")